# Usage metering tests

In [1]:
import boto3
import urllib.parse as urlparse 
import json
import shutil
import time

from datetime import datetime, timezone

In [2]:
PROFILE = 'default'
REGION = 'us-east-1'

SESSION = boto3.Session(profile_name=PROFILE, region_name=REGION)

In [22]:

def get_marketplace_product(product_id):
    client = SESSION.client('marketplace-catalog')

    response = client.describe_entity(
    Catalog='AWSMarketplace',
    EntityId=product_id
    )
    
    return response

def create_metering_item(customer_aws_account_id, dimension_name):
    item = {
        "create_timestamp": {
            "N": f"{int(time.time())}"
        },
        "customerIdentifier": {
            "S": customer_aws_account_id
        },
        "dimension_usage": {
            "L": [
            {
                "M": {
                "dimension": {
                    "S": dimension_name
                },
                "value": {
                    "N": "3"
                }
                }
            }
            ]
        },
        "metering_pending": {
            "S": "true"
        }
    }
    
    return item


def scan_table(table_name):
    dynamodb = boto3.resource('dynamodb')
    table = dynamodb.Table(table_name)
    
    response = table.scan()
    return response['Items']


## Put metering recoreds into DynamoDB

Put metering records (dimensions) into the metering table. You can get the metering table name from your CFN stack.

### Get DynamoDB table names
Get DynamoDB table names from the SaaS integration CFN stack.

Replace the value for `stack_name` with the name of you CFN stack.

Use `Metering table name` to ingest metering records.

In [16]:
stack_name = 'eb-saas-sub'
cf_client = boto3.client('cloudformation')
paginator = cf_client.get_paginator('list_stack_resources')

for page in paginator.paginate(StackName=stack_name):
    for resource in page['StackResourceSummaries']:
        #print(json.dumps(resource, indent=2, default=str))
        #print(f"{resource['ResourceType']}: {resource['LogicalResourceId']}")
        if resource['LogicalResourceId'] == 'AWSMarketplaceSubscribers':
            print(f"Subscribers table name: ${resource['PhysicalResourceId']}")
        if resource['LogicalResourceId'] == 'AWSMarketplaceMeteringRecords':
            print(f"Metering table name: ${resource['PhysicalResourceId']}")

Metering table name: $MPMeteringSub
Subscribers table name: $MPSubscribersSub


In [23]:
# Replace the table name with the table from your environment
metering_table_name = 'MPMeteringSub'
ddb = SESSION.client('dynamodb')

## Create metering entries

Use `create_metering_item(CustomerAWSAccounId, Dimension)` to create
item and put them into the DynamoDB metering table.

In [30]:
# replace the aws account id and the dimension with your settings
item = create_metering_item('944681004585', 'usage_2')
print(json.dumps(item, indent=2, default=str))

response = ddb.put_item(
    TableName=metering_table_name,
    Item=item
)
print(json.dumps(response, indent=2, default=str))

{
  "create_timestamp": {
    "N": "1763396979"
  },
  "customerIdentifier": {
    "S": "944681004585"
  },
  "dimension_usage": {
    "L": [
      {
        "M": {
          "dimension": {
            "S": "usage_2"
          },
          "value": {
            "N": "3"
          }
        }
      }
    ]
  },
  "metering_pending": {
    "S": "true"
  }
}
{
  "ResponseMetadata": {
    "RequestId": "QLRM5D6635KLLB2VDTHLA83DRFVV4KQNSO5AEMVJF66Q9ASUAAJG",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "server": "Server",
      "date": "Mon, 17 Nov 2025 16:29:39 GMT",
      "content-type": "application/x-amz-json-1.0",
      "content-length": "2",
      "connection": "keep-alive",
      "x-amzn-requestid": "QLRM5D6635KLLB2VDTHLA83DRFVV4KQNSO5AEMVJF66Q9ASUAAJG",
      "x-amz-crc32": "2745614147"
    },
    "RetryAttempts": 0
  }
}


## Get entries from the metering table

Scan the metering table.

Unprocessed item look similar to:

```
{
  "dimension_usage": [
    {
      "dimension": "usage_2",
      "value": "3"
    }
  ],
  "metering_pending": "true",
  "create_timestamp": "1763396941",
  "customerIdentifier": "944681004585"
}
```

Processed records have a `metering_failed` boolean key and a `metering_response` key for example:

```
"metering_failed": false,
  "dimension_usage": [
    {
      "dimension": "usage_1",
      "value": "3"
    }
  ],
  "create_timestamp": "1763392067",
  "customerIdentifier": "944681004585",
  "metering_response": "{\"$metadata\":{\"httpStatusCode\":200,\"requestId\":\"828fd9e5-ee21-4b0f-9656-82a86b4e49c2\",\"attempts\":1,\"totalRetryDelay\":0},\"Results\":[{\"MeteringRecordId\":\"eb0e9db9-c90e-45fa-84c4-6239a68fb360\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_1\",\"Quantity\":3,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}},{\"MeteringRecordId\":\"6756c73c-21fd-4821-a5d5-57a6d891f103\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_2\",\"Quantity\":6,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}},{\"MeteringRecordId\":\"c65d412f-aedf-4e89-afa3-a1d238277b63\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_3\",\"Quantity\":3,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}}],\"UnprocessedRecords\":[]}"
}
```

In [31]:
# Usage
items = scan_table(metering_table_name)
for item in items:
    print(json.dumps(item, indent=2, default=str))
    print('-' * 50)

{
  "metering_failed": false,
  "dimension_usage": [
    {
      "dimension": "usage_1",
      "value": "3"
    }
  ],
  "create_timestamp": "1763392067",
  "customerIdentifier": "944681004585",
  "metering_response": "{\"$metadata\":{\"httpStatusCode\":200,\"requestId\":\"828fd9e5-ee21-4b0f-9656-82a86b4e49c2\",\"attempts\":1,\"totalRetryDelay\":0},\"Results\":[{\"MeteringRecordId\":\"eb0e9db9-c90e-45fa-84c4-6239a68fb360\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_1\",\"Quantity\":3,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}},{\"MeteringRecordId\":\"6756c73c-21fd-4821-a5d5-57a6d891f103\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_2\",\"Quantity\":6,\"Timestamp\":\"2025-11-17T16:01:34.018Z\"}},{\"MeteringRecordId\":\"c65d412f-aedf-4e89-afa3-a1d238277b63\",\"Status\":\"Success\",\"UsageRecord\":{\"CustomerAWSAccountId\":\"944681004585\",\"Dimension\":\"usage_3\",\"Q